<a href="https://colab.research.google.com/github/vanadhisivakumar-source/Machine-learning-projects/blob/main/house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 1. Dataset Preparation
First, let's define the house size and price data as a pandas DataFrame.

In [ ]:
data = {
    'Size (sq.ft)': [800, 1000, 1200, 1500, 1800, 2000],
    'Price (₹ Lakhs)': [20, 25, 30, 38, 45, 50]
}
df = pd.DataFrame(data)

X_train = df['Size (sq.ft)'].values
Y_train = df['Price (₹ Lakhs)'].values

display(df)

### 2. Locally Weighted Regression Implementation
Now, let's implement the core components of the LWR algorithm:

1.  **Gaussian Kernel Function:** This function will assign weights to training points based on their distance from the query point.
2.  **Weighted Linear Regression:** For each query, we'll perform a linear regression where each training point's contribution is scaled by its weight.

In [ ]:
def gaussian_kernel(query_point, training_points, tau):
    """
    Computes the Gaussian kernel weights for training points relative to a query point.
    Args:
        query_point (float): The point for which we want to predict.
        training_points (np.array): Array of training feature values.
        tau (float): The bandwidth parameter, controlling the width of the kernel.
    Returns:
        np.array: Array of weights for each training point.
    """
    diff = training_points - query_point
    weights = np.exp(-(diff**2) / (2 * tau**2))
    return weights

def locally_weighted_regression(X_train, Y_train, query_point, tau):
    """
    Performs Locally Weighted Regression to predict the target for a query point.
    Args:
        X_train (np.array): Training feature values (house sizes).
        Y_train (np.array): Training target values (house prices).
        query_point (float): The feature value for which to make a prediction.
        tau (float): The bandwidth parameter for the Gaussian kernel.
    Returns:
        float: The predicted target value (house price).
    """
    # Add a bias (intercept) term to X_train
    X_bias = np.vstack([np.ones_like(X_train), X_train]).T

    # Calculate weights using the Gaussian kernel
    weights = gaussian_kernel(query_point, X_train, tau)
    W = np.diag(weights)

    # Perform weighted linear regression
    # (X_T * W * X)^-1 * X_T * W * Y
    try:
        term1 = X_bias.T @ W @ X_bias
        term2 = X_bias.T @ W @ Y_train
        theta = np.linalg.inv(term1) @ term2
    except np.linalg.LinAlgError:
        # Handle cases where (X_T * W * X) might be singular (e.g., all weights are zero)
        # This can happen if tau is too small and query_point is far from all X_train points
        # In such cases, we can return a default or nearest neighbor value, or raise an error.
        # For simplicity here, we'll just return a mean or interpolation.
        print(f"Warning: Singular matrix for query_point={query_point}. Returning mean of Y_train.")
        return np.mean(Y_train)

    # Predict for the query point
    query_bias = np.array([1, query_point])
    prediction = query_bias @ theta
    return prediction

### 3. Test Example Prediction
Now, let's use our implemented LWR algorithm to predict the price for a house of Size = 1400 sq.ft.

In [ ]:
test_size = 1400  # sq.ft
tau_parameter = 100 # Bandwidth parameter (tune this for desired locality)

predicted_price = locally_weighted_regression(X_train, Y_train, test_size, tau_parameter)

print(f"Predicted price for a house of size {test_size} sq.ft: ₹ {predicted_price:.2f} Lakhs")

### 4. Visualization (Optional)
Let's visualize the training data and the predicted point. We can also plot the LWR prediction curve over a range of sizes to see how it fits the data.

In [ ]:
fig = plt.figure(figsize=(10, 6))
plt.scatter(X_train, Y_train, color='blue', label='Training Data')
plt.scatter(test_size, predicted_price, color='red', marker='x', s=100, label=f'Predicted Price for {test_size} sq.ft')

# To visualize the LWR curve, we can predict for a range of X values
X_range = np.linspace(min(X_train) - 100, max(X_train) + 100, 100)
Y_pred_lwr = [locally_weighted_regression(X_train, Y_train, x, tau_parameter) for x in X_range]
plt.plot(X_range, Y_pred_lwr, color='green', linestyle='--', label=f'LWR Prediction Curve (tau={tau_parameter})')

plt.title('House Price Prediction using Locally Weighted Regression')
plt.xlabel('House Size (sq.ft)')
plt.ylabel('Price (₹ Lakhs)')
plt.legend()
plt.grid(True)
plt.show()